# Manga Recap Render Server (v3)

Jalankan 3 sel di bawah **berurutan** (tekan play tiap sel, tunggu selesai baru lanjut).

**Disarankan pakai GPU** biar sync otomatis cepat: menu *Runtime > Change runtime type > T4 GPU*. Tanpa GPU tetap jalan, tapi transkripsi audio panjang jauh lebih lambat.

Setelah sel 3 selesai akan muncul **Link**. Buka di browser (HP atau laptop), di sana ada 4 tab:

1. **Gambar**: drag and drop panel (atau tap untuk pilih), atur urutan dengan tombol.
2. **Naskah**: tempel naskah, satu paragraf (dipisah baris kosong) = satu gambar.
3. **Sync**: pilih audio narasi lalu tekan **Sync otomatis**. Audio ditranskrip (Whisper) dan dicocokkan ke tiap paragraf. Panel bertanda ⚠ perlu dicek, tap ▶ buat dengar hasilnya, durasi bisa dikoreksi manual. **Tap-sync manual** tetap tersedia sebagai cadangan.
4. **Render**: pilih gerakan, ukuran, transisi, lalu Render. Video tanpa subtitle, bisa langsung diputar dan diunduh.

Catatan: pemakaian pertama sync otomatis agak lama karena mengunduh model Whisper. Jangan tutup tab Colab selama dipakai.

In [3]:
# Sel 1: pasang kebutuhan (sekali tiap sesi baru)
!pip -q install fastapi uvicorn python-multipart pillow faster-whisper
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!ffmpeg -version | head -1
!nvidia-smi -L || echo "GPU tidak aktif (CPU saja)"
print("Siap")

/content/cloudflared: Text file busy
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
GPU 0: Tesla T4 (UUID: GPU-72c1b9d4-3389-742b-a1f0-a2d1043fcce8)
Siap


In [4]:
%%writefile /content/recap_server.py
import os, sys, json, uuid, random, subprocess, threading, traceback, functools
from pathlib import Path
from typing import List, Optional

from PIL import Image

WORK = Path("/content/recap_jobs") if os.path.isdir("/content") else Path("/tmp/recap_jobs")
WORK.mkdir(parents=True, exist_ok=True)
JOBS = {}

PAN_MOTIONS = ["pan-left-to-right", "pan-right-to-left", "pan-top-to-bottom", "pan-bottom-to-top"]


# ---------- gerakan panel ----------
def resolve_motion(m):
    m = (m or "").lower()
    if "random" in m or "smart" in m or "multi" in m or "flow" in m:
        return random.choice(PAN_MOTIONS + ["zoom-in", "zoom-out"])
    if "top-to-bottom" in m:
        return "pan-top-to-bottom"
    if "bottom-to-top" in m:
        return "pan-bottom-to-top"
    if "left-to-right" in m:
        return "pan-left-to-right"
    if "right-to-left" in m:
        return "pan-right-to-left"
    if "shaky" in m:
        return "shaky-action"
    if "zoom-out" in m:
        return "zoom-out"
    return "zoom-in"


def ease(p, kind="smooth"):
    p = min(max(p, 0.0), 1.0)
    return p if kind == "linear" else p * p * (3 - 2 * p)


def window_box(iw, ih, W, H, motion, p, speed, kind):
    ar = W / H
    if iw / ih > ar:
        bh = ih
        bw = bh * ar
    else:
        bw = iw
        bh = bw / ar
    e = ease(p, kind)
    k = min(max(speed, 0.1), 3.0)
    pan_e = 0.5 + (e - 0.5) * min(k, 1.0)  # speed kecil = geser lebih pendek
    z = 0.72
    amt = min(0.38 * k, 0.6)
    if motion == "zoom-in":
        z = 1.0 - amt * e
    elif motion == "zoom-out":
        z = 1.0 - amt * (1 - e)
    elif motion == "shaky-action":
        z = 0.82
    ww, wh = bw * z, bh * z
    rx, ry = iw - ww, ih - wh
    x, y = rx / 2, ry / 2
    if motion == "pan-left-to-right":
        x = rx * pan_e
    elif motion == "pan-right-to-left":
        x = rx * (1 - pan_e)
    elif motion == "pan-top-to-bottom":
        y = ry * pan_e
    elif motion == "pan-bottom-to-top":
        y = ry * (1 - pan_e)
    elif motion == "shaky-action":
        rnd = random.Random(int(p * 100000))
        x += (rnd.random() - 0.5) * iw * 0.03
        y += (rnd.random() - 0.5) * ih * 0.03
    x = min(max(x, 0), max(rx, 0))
    y = min(max(y, 0), max(ry, 0))
    return (x, y, x + ww, y + wh)


def make_frame(img, plan, p, W, H):
    box = window_box(img.width, img.height, W, H, plan["motion"], p, plan["speed"], plan["easing"])
    return img.resize((W, H), Image.BICUBIC, box=box)


def probe_duration(path):
    try:
        r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                            "-of", "default=nw=1:nk=1", str(path)],
                           capture_output=True, text=True)
        return float(r.stdout.strip())
    except Exception:
        return 0.0


# ---------- render ----------
def render_job(job_id, panel_paths, audio_path, cfg):
    job = JOBS[job_id]
    try:
        job["status"] = "rendering"
        W = int(cfg.get("width", 1280))
        H = int(cfg.get("height", 720))
        W -= W % 2
        H -= H % 2
        fps = int(cfg.get("fps", 24))
        trans = float(cfg.get("transition_sec", 0.4))
        scenes = cfg.get("scenes", []) or []

        imgs, plans = [], []
        for i, pth in enumerate(panel_paths):
            im = Image.open(pth).convert("RGB")
            im.thumbnail((3000, 3000))
            imgs.append(im)
            sc = scenes[i] if i < len(scenes) else {}
            plans.append({
                "dur": max(0.2, float(sc.get("duration", 4))),
                "motion": resolve_motion(sc.get("motionType")),
                "speed": float(sc.get("motionSpeed", 1.0) or 1.0),
                "easing": sc.get("easing", "smooth"),
                "trans": sc.get("transitionType", "crossfade"),
            })

        # kalau audio lebih panjang dari total durasi panel, panel terakhir diperpanjang
        if audio_path:
            alen = probe_duration(audio_path)
            tot = sum(pl["dur"] for pl in plans)
            if alen > tot and plans:
                plans[-1]["dur"] += alen - tot

        # jumlah frame dihitung dari waktu kumulatif supaya tidak geser (drift)
        bounds, acc = [0], 0.0
        for pl in plans:
            acc += pl["dur"]
            bounds.append(int(round(acc * fps)))
        nfs = [max(1, bounds[i + 1] - bounds[i]) for i in range(len(plans))]
        total = sum(nfs)

        out_dir = WORK / job_id
        silent = out_dir / "silent.mp4"
        final = out_dir / "final.mp4"
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24",
               "-s", f"{W}x{H}", "-r", str(fps), "-i", "-", "-c:v", "libx264",
               "-preset", "veryfast", "-crf", "23", "-pix_fmt", "yuv420p",
               "-movflags", "+faststart", str(silent)]
        proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)

        done, last = 0, None
        for i, pl in enumerate(plans):
            nf = nfs[i]
            tr = min(trans, nf / fps * 0.5)
            for f in range(nf):
                p = f / max(1, nf - 1)
                fr = make_frame(imgs[i], pl, p, W, H)
                if last is not None and pl["trans"] != "none" and tr > 0 and f / fps < tr:
                    fr = Image.blend(last, fr, (f / fps) / tr)
                proc.stdin.write(fr.tobytes())
                done += 1
                if done % 12 == 0:
                    job["progress"] = round(done / total * 0.9, 3)
            last = make_frame(imgs[i], pl, 1.0, W, H)
        proc.stdin.close()
        if proc.wait() != 0:
            raise RuntimeError("ffmpeg gagal merender video")

        if audio_path:
            job["progress"] = 0.93
            r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(silent), "-i", str(audio_path),
                                "-map", "0:v", "-map", "1:a", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                                str(final)], capture_output=True, text=True)
            if r.returncode != 0:
                raise RuntimeError("gagal menempel audio: " + r.stderr[-300:])
        else:
            silent.rename(final)
        job["progress"] = 1.0
        job["status"] = "done"
    except Exception as e:
        traceback.print_exc()
        job["status"] = "error"
        job["error"] = str(e)



# ---------- sync otomatis: cocokkan transkrip ke paragraf ----------
import re, difflib


def _norm(w):
    return re.sub(r"[^\w]+", "", (w or "").lower(), flags=re.UNICODE)


def align_paragraphs(words, paras, audio_len, lead=0.15, min_dur=0.3):
    """words: [{'w':str,'s':float,'e':float}, ...] dari Whisper.
    paras: list teks paragraf (satu paragraf = satu panel).
    Mengembalikan (starts, durations, conf). conf = porsi kata paragraf yang cocok (0..1)."""
    tw = [(_norm(x["w"]), x["s"]) for x in words]
    tw = [(t, s) for t, s in tw if t]
    ttok = [t for t, _ in tw]
    ttime = [s for _, s in tw]

    stok, spar, first_idx = [], [], []
    for pi, p in enumerate(paras):
        toks = [t for t in (_norm(x) for x in p.split()) if t]
        if not toks:
            toks = ["\x00%d" % pi]  # placeholder yang tidak akan cocok
        first_idx.append(len(stok))
        stok.extend(toks)
        spar.extend([pi] * len(toks))

    # titik cocok antara indeks naskah dan indeks transkrip
    match = {}
    if ttok:
        sm = difflib.SequenceMatcher(None, stok, ttok, autojunk=False)
        for blk in sm.get_matching_blocks():
            for k in range(blk.size):
                si, ti = blk.a + k, blk.b + k
                # jangkar andal: blok >= 2 kata, atau kata panjang
                if blk.size >= 2 or len(stok[si]) >= 5:
                    match[si] = ttime[ti]

    # kepercayaan per paragraf
    cnt = [0] * len(paras)
    tot = [0] * len(paras)
    for si, pi in enumerate(spar):
        tot[pi] += 1
        if si in match:
            cnt[pi] += 1
    conf = [round(cnt[i] / tot[i], 2) if tot[i] else 0.0 for i in range(len(paras))]

    anchors = sorted(match.items())  # [(idx naskah, waktu)]
    n_tok = max(1, len(stok))
    if len(anchors) >= 2:
        rate = (anchors[-1][1] - anchors[0][1]) / max(1, anchors[-1][0] - anchors[0][0])
    else:
        rate = audio_len / n_tok
    rate = min(max(rate, 0.1), 1.5)

    def t_at(k):
        if not anchors:
            return audio_len * k / n_tok
        if k <= anchors[0][0]:
            sa, ta = anchors[0]
            return max(0.0, ta - (sa - k) * rate)
        if k >= anchors[-1][0]:
            sb, tb = anchors[-1]
            return min(audio_len, tb + (k - sb) * rate)
        lo, hi = 0, len(anchors) - 1
        while hi - lo > 1:
            mid = (lo + hi) // 2
            if anchors[mid][0] <= k:
                lo = mid
            else:
                hi = mid
        (sa, ta), (sb, tb) = anchors[lo], anchors[hi]
        if k == sa:
            return ta
        return ta + (k - sa) / (sb - sa) * (tb - ta)

    starts = [0.0]
    for i in range(1, len(paras)):
        t = t_at(first_idx[i]) - lead
        t = max(t, starts[-1] + min_dur)
        starts.append(round(t, 2))
    # jaga supaya semua panel masih muat di dalam audio
    for i in range(len(starts) - 1, -1, -1):
        limit = audio_len - min_dur * (len(starts) - i)
        if starts[i] > limit:
            starts[i] = round(max(limit, 0.0), 2)
    durs = []
    for i in range(len(starts)):
        e = starts[i + 1] if i + 1 < len(starts) else audio_len
        durs.append(round(max(min_dur, e - starts[i]), 2))
    return starts, durs, conf


WORKER_CODE = r"""
import sys, json
from faster_whisper import WhisperModel
audio, out, model, device, lang = sys.argv[1:6]
ctype = "float16" if device == "cuda" else "int8"
m = WhisperModel(model, device=device, compute_type=ctype)
segs, info = m.transcribe(audio, language=None if lang == "auto" else lang,
                          word_timestamps=True, beam_size=5 if device == "cuda" else 1,
                          condition_on_previous_text=False)
dur = float(info.duration or 0) or 1.0
words = []
for sg in segs:
    for w in (sg.words or []):
        words.append({"w": w.word, "s": round(w.start, 2), "e": round(w.end, 2)})
    print("PROGRESS %.3f" % min(sg.end / dur, 1.0), flush=True)
json.dump({"words": words, "duration": dur, "language": info.language},
          open(out, "w", encoding="utf-8"))
print("DONE", flush=True)
"""


@functools.lru_cache(maxsize=1)
def has_gpu():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception:
        return False


def run_worker(audio_path, out_json, model, device, lang, job):
    wfile = WORK / "transcribe_worker.py"
    wfile.write_text(WORKER_CODE, encoding="utf-8")
    proc = subprocess.Popen([sys.executable, str(wfile), str(audio_path), str(out_json), model, device, lang],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in proc.stdout:
        line = line.strip()
        if line.startswith("PROGRESS"):
            try:
                job["progress"] = round(float(line.split()[1]) * 0.9, 3)
            except Exception:
                pass
        elif line:
            tail = (tail + [line])[-8:]
    return proc.wait(), "\n".join(tail)


def autosync_job(job_id, audio_path, paras, model, lang):
    job = JOBS[job_id]
    try:
        job["status"] = "transcribing"
        out = WORK / job_id / "words.json"
        device = "cuda" if has_gpu() else "cpu"
        rc, tail = run_worker(audio_path, out, model, device, lang, job)
        if rc != 0 and device == "cuda":
            job["note"] = "GPU gagal dipakai, pindah ke CPU (lebih lambat)"
            device = "cpu"
            if model == "medium":
                model = "small"
            job["progress"] = 0.0
            rc, tail = run_worker(audio_path, out, model, device, lang, job)
        if rc != 0 or not out.exists():
            raise RuntimeError("transkripsi gagal: " + tail[-300:])
        data = json.load(open(out, encoding="utf-8"))
        if not data.get("words"):
            raise RuntimeError("tidak ada suara yang terdeteksi di audio")
        job["status"] = "aligning"
        job["progress"] = 0.95
        alen = max(probe_duration(audio_path), float(data.get("duration", 0)))
        starts, durs, conf = align_paragraphs(data["words"], paras, alen)
        job.update(durations=durs, starts=starts, conf=conf, device=device,
                   language=data.get("language"), words=len(data["words"]))
        job["progress"] = 1.0
        job["status"] = "done"
    except Exception as e:
        traceback.print_exc()
        job["status"] = "error"
        job["error"] = str(e)


# ---------- API ----------
TEST_PAGE = r"""<!doctype html>
<html lang="id"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Manga Recap Maker</title>
<style>
:root{--bg:#14161a;--card:#1e2128;--line:#2e333d;--txt:#e9ecf1;--mut:#9aa3b2;--acc:#ff6b4a;--ok:#3ecf8e;--warn:#f5b942}
*{box-sizing:border-box}
[hidden]{display:none!important}
body{margin:0;background:var(--bg);color:var(--txt);font:16px/1.4 system-ui,sans-serif;padding-bottom:110px}
header{padding:12px 14px;font-weight:700;border-bottom:1px solid var(--line)}
nav{display:flex;position:sticky;top:0;background:var(--bg);z-index:5;border-bottom:1px solid var(--line)}
nav button{flex:1;padding:12px 4px;background:none;border:0;border-bottom:3px solid transparent;color:var(--mut);font-size:14px;font-weight:600}
nav button.on{color:var(--txt);border-color:var(--acc)}
main{max-width:640px;margin:0 auto;padding:12px}
.card{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:12px;margin-bottom:12px}
label{display:block;font-size:13px;color:var(--mut);margin:10px 0 4px}
input,select,textarea,button{font:inherit;color:var(--txt)}
input[type=number],input[type=file],select,textarea{width:100%;padding:10px;background:#12141a;border:1px solid var(--line);border-radius:8px}
textarea{min-height:240px;resize:vertical}
button{touch-action:manipulation}
.btn{padding:12px 16px;border-radius:10px;border:0;background:var(--acc);color:#fff;font-weight:700;width:100%}
.btn.sec{background:#2a2f39}
.mut{color:var(--mut);font-size:13px}
.ok{color:var(--ok)}.warn{color:var(--warn)}
#drop{border:2px dashed var(--line);border-radius:12px;padding:26px 12px;text-align:center;color:var(--mut)}
#drop.over{border-color:var(--acc);color:var(--txt)}
#grid{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin-top:12px}
.th{position:relative;border:1px solid var(--line);border-radius:8px;overflow:hidden;background:#000}
.th img{width:100%;aspect-ratio:3/4;object-fit:cover;display:block}
.th .n{position:absolute;top:4px;left:4px;background:var(--acc);border-radius:6px;padding:0 6px;font-size:12px;font-weight:700}
.th .ct{display:flex}
.th .ct button{flex:1;padding:8px 0;background:#2a2f39;border:0;border-right:1px solid var(--line);font-size:13px}
.th .ct button:last-child{border:0}
.th.drag{opacity:.4}
.para{padding:8px 0;border-bottom:1px solid var(--line);font-size:14px}
.para b{color:var(--acc)}
#stage img{width:100%;max-height:46vh;object-fit:contain;background:#000;border-radius:8px}
#curtxt{font-size:18px;margin:10px 0 6px}
#clock{font-variant-numeric:tabular-nums;color:var(--mut);margin-top:6px}
.row{display:flex;align-items:center;gap:8px;padding:6px 0;border-bottom:1px solid var(--line)}
.row img{width:36px;height:48px;object-fit:cover;border-radius:4px}
.row span{flex:1;font-size:14px;min-width:0}
.row small{display:block;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.row .pv{width:44px;height:44px;border:0;border-radius:8px;background:#2a2f39;font-size:16px}
.row input{width:90px}
#bar{position:fixed;left:0;right:0;bottom:0;background:var(--bg);border-top:1px solid var(--line);padding:10px;display:flex;gap:8px;z-index:9}
#bar button{border:0;border-radius:12px;font-weight:700;font-size:17px}
#bTap{flex:1;height:64px;background:var(--acc)}
#bUndo,#bPause{width:64px;height:64px;background:#2a2f39}
video{width:100%;margin-top:10px;border-radius:8px;background:#000}
progress{width:100%}
#toast{position:fixed;left:12px;right:12px;top:60px;background:#000c;border:1px solid var(--line);border-radius:10px;padding:10px;z-index:20;text-align:center}
</style></head>
<body>
<header>Manga Recap Maker</header>
<nav id="tabs">
  <button data-t="img" class="on">1 Gambar</button>
  <button data-t="txt">2 Naskah</button>
  <button data-t="sync">3 Sync</button>
  <button data-t="out">4 Render</button>
</nav>
<main>

<section id="t-img">
  <div class="card">
    <div id="drop">Drag &amp; drop gambar panel ke sini<br><span class="mut">atau tap untuk pilih (bisa banyak)</span></div>
    <input type="file" id="pf" multiple accept="image/*" hidden>
    <div id="cnt" class="mut" style="margin-top:10px"></div>
    <div id="grid"></div>
    <p class="mut">Urutan gambar = urutan panel. Atur dengan tombol ◀ ▶ (di desktop bisa drag antar gambar), ✕ untuk hapus.</p>
    <button class="btn sec" id="bClear">Hapus semua gambar</button>
  </div>
</section>

<section id="t-txt" hidden>
  <div class="card">
    <label>Naskah narasi. Pisahkan paragraf dengan baris kosong. Satu paragraf = satu gambar.</label>
    <textarea id="script" placeholder="Paragraf 1...&#10;&#10;Paragraf 2...&#10;&#10;Paragraf 3..."></textarea>
    <div id="match" style="margin-top:8px"></div>
  </div>
  <div class="card" id="plist"></div>
</section>

<section id="t-sync" hidden>
  <div class="card">
    <label>Audio narasi</label>
    <input type="file" id="af" accept="audio/*">
  </div>

  <div class="card">
    <b>Sync otomatis</b>
    <div class="mut">Audio ditranskrip (Whisper) lalu dicocokkan ke tiap paragraf naskah. Jumlah paragraf harus sama dengan jumlah gambar.</div>
    <label>Bahasa narasi</label>
    <select id="lang">
      <option value="id">Indonesia</option>
      <option value="en">English</option>
      <option value="auto">Deteksi otomatis</option>
    </select>
    <label>Model (makin besar makin akurat tapi makin lama)</label>
    <select id="wm">
      <option value="small">small (disarankan)</option>
      <option value="medium">medium (paling akurat)</option>
      <option value="base">base (cepat)</option>
    </select>
    <div id="srv" class="mut" style="margin:8px 0"></div>
    <button class="btn" id="bAuto">Sync otomatis</button>
    <progress id="apg" value="0" max="1" style="margin-top:10px" hidden></progress>
    <div id="ast" class="mut"></div>
  </div>

  <button class="btn sec" id="bManual" style="margin-bottom:12px">Tap-sync manual (opsional)</button>
  <div id="manual" hidden>
    <div class="card">
      <label>Koreksi telat tap (detik). Tap biasanya telat sekitar 0.2 detik, nilai ini dikurangkan otomatis.</label>
      <input type="number" id="lag" value="0.2" step="0.05" min="0">
    </div>
    <div class="card" id="stage">
      <div class="mut" id="curn">Belum ada gambar</div>
      <img id="cur" alt="" hidden>
      <div id="curtxt"></div>
      <div id="nexttxt" class="mut"></div>
      <div id="clock">0:00 / 0:00</div>
    </div>
  </div>

  <div class="card" id="durbox" hidden>
    <b>Durasi tiap panel (detik)</b>
    <div class="mut">Tap ▶ buat dengar bagian audio panel itu. Angka bisa dikoreksi manual. Tanda ⚠ = kecocokan dengan naskah rendah, perlu dicek.</div>
    <div id="durlist"></div>
    <div id="durtot" class="mut" style="margin:8px 0"></div>
    <button class="btn sec" id="bEven">Bagi rata sesuai panjang audio</button>
  </div>
</section>

<section id="t-out" hidden>
  <div class="card">
    <label>Gerakan panel</label>
    <select id="pm">
      <option value="random">Acak (campur)</option>
      <option value="pan-top-to-bottom">Geser atas ke bawah</option>
      <option value="pan-bottom-to-top">Geser bawah ke atas</option>
      <option value="pan-left-to-right">Geser kiri ke kanan</option>
      <option value="pan-right-to-left">Geser kanan ke kiri</option>
      <option value="zoom-in">Zoom in</option>
      <option value="zoom-out">Zoom out</option>
      <option value="shaky-action">Getar (action)</option>
    </select>
    <label>Ukuran</label>
    <select id="ps">
      <option value="1280x720">16:9 (YouTube)</option>
      <option value="720x1280">9:16 (Shorts/Reels)</option>
    </select>
    <label>Transisi antar panel</label>
    <select id="ptr">
      <option value="0.4">Crossfade</option>
      <option value="0">Tanpa transisi</option>
    </select>
    <div id="chk" class="mut" style="margin:12px 0"></div>
    <button class="btn" id="bGo">Render video</button>
    <progress id="pg" value="0" max="1" style="margin-top:12px"></progress>
    <div id="st" class="mut"></div>
    <video id="pv" controls playsinline hidden></video>
    <a id="dl" class="btn" style="display:none;text-align:center;text-decoration:none;margin-top:10px" download="recap.mp4">Unduh video</a>
  </div>
</section>

</main>

<div id="bar" hidden>
  <button id="bUndo" title="Mundur satu panel">&#8630;</button>
  <button id="bTap">Mulai</button>
  <button id="bPause">&#9208;</button>
</div>
<div id="toast" hidden></div>

<script>
const $ = id => document.getElementById(id);
let panels = [];          // {file, url}
let paras = [];           // paragraf naskah
let durs = [];            // detik per panel
let audioFile = null;
const aud = new Audio();
aud.preload = 'auto';
let state = 'idle';       // idle | playing | paused | done
let idx = 0;
let starts = [];
let wake = null;
let curTab = 'img';
let manualOpen = false;
let lowconf = [];
let pvTimer = null;
function clearDurs() { durs = []; lowconf = []; }

function toast(m, ms) {
  const t = $('toast'); t.textContent = m; t.hidden = false;
  clearTimeout(toast.h); toast.h = setTimeout(() => t.hidden = true, ms || 2600);
}
function fmt(s) {
  s = Math.max(0, s || 0);
  const m = Math.floor(s / 60), r = Math.floor(s % 60);
  return m + ':' + String(r).padStart(2, '0');
}
function snip(s, n) { s = s || ''; return s.length > n ? s.slice(0, n) + '…' : s; }

/* ---------- tab ---------- */
document.querySelectorAll('#tabs button').forEach(b => b.onclick = () => showTab(b.dataset.t));
function showTab(t) {
  curTab = t;
  document.querySelectorAll('#tabs button').forEach(b => b.classList.toggle('on', b.dataset.t === t));
  ['img', 'txt', 'sync', 'out'].forEach(k => $('t-' + k).hidden = (k !== t));
  $('bar').hidden = !(t === 'sync' && manualOpen);
  if (t === 'sync') renderSync();
  if (t === 'out') renderCheck();
}

/* ---------- gambar ---------- */
const drop = $('drop'), pf = $('pf');
drop.onclick = () => pf.click();
pf.onchange = () => { addFiles(pf.files); pf.value = ''; };
['dragenter', 'dragover'].forEach(ev => drop.addEventListener(ev, e => { e.preventDefault(); drop.classList.add('over'); }));
['dragleave', 'drop'].forEach(ev => drop.addEventListener(ev, e => { e.preventDefault(); drop.classList.remove('over'); }));
drop.addEventListener('drop', e => { if (e.dataTransfer && e.dataTransfer.files.length) addFiles(e.dataTransfer.files); });
// cegah browser membuka file kalau ter-drop di luar kotak
['dragover', 'drop'].forEach(ev => window.addEventListener(ev, e => { if (!e.target.closest || !e.target.closest('.th')) e.preventDefault(); }));

function addFiles(list) {
  const fs = [...list].filter(f => f.type.startsWith('image/'))
    .sort((a, b) => a.name.localeCompare(b.name, undefined, { numeric: true }));
  if (!fs.length) { toast('Tidak ada file gambar'); return; }
  fs.forEach(f => panels.push({ file: f, url: URL.createObjectURL(f) }));
  panelsChanged();
}
function panelsChanged() {
  if (durs.length !== panels.length) clearDurs();
  if (state !== 'idle') resetSync();
  renderGrid(); renderMatch(); renderSync(); renderCheck();
}
let dragIdx = null;
function renderGrid() {
  const g = $('grid'); g.innerHTML = '';
  panels.forEach((p, i) => {
    const d = document.createElement('div');
    d.className = 'th'; d.dataset.i = i; d.draggable = true;
    d.innerHTML = '<img src="' + p.url + '" alt=""><span class="n">' + (i + 1) + '</span>' +
      '<div class="ct"><button data-a="l">◀</button><button data-a="r">▶</button><button data-a="x">✕</button></div>';
    d.addEventListener('dragstart', () => { dragIdx = i; d.classList.add('drag'); });
    d.addEventListener('dragend', () => { dragIdx = null; d.classList.remove('drag'); });
    d.addEventListener('dragover', e => { if (dragIdx !== null) e.preventDefault(); });
    d.addEventListener('drop', e => {
      if (dragIdx === null) return;
      e.preventDefault();
      move(dragIdx, i); dragIdx = null;
    });
    g.appendChild(d);
  });
  $('cnt').textContent = panels.length ? panels.length + ' gambar' : 'Belum ada gambar';
}
function move(a, b) {
  if (b < 0 || b >= panels.length || a === b) return;
  const [x] = panels.splice(a, 1); panels.splice(b, 0, x);
  clearDurs(); panelsChanged();
}
$('grid').addEventListener('click', e => {
  const a = e.target.dataset && e.target.dataset.a;
  if (!a) return;
  const i = +e.target.closest('.th').dataset.i;
  if (a === 'l') move(i, i - 1);
  else if (a === 'r') move(i, i + 1);
  else if (a === 'x') { URL.revokeObjectURL(panels[i].url); panels.splice(i, 1); clearDurs(); panelsChanged(); }
});
$('bClear').onclick = () => {
  if (!panels.length || !confirm('Hapus semua gambar?')) return;
  panels.forEach(p => URL.revokeObjectURL(p.url)); panels = []; clearDurs(); panelsChanged();
};

/* ---------- naskah ---------- */
$('script').addEventListener('input', parseScript);
function parseScript() {
  paras = $('script').value.split(/\n\s*\n/).map(s => s.trim()).filter(Boolean);
  renderMatch(); renderSync();
}
function renderMatch() {
  const m = $('match');
  if (!paras.length) m.innerHTML = '<span class="mut">Belum ada naskah. Gambar: ' + panels.length + '</span>';
  else if (paras.length === panels.length) m.innerHTML = '<span class="ok">✔ ' + paras.length + ' paragraf = ' + panels.length + ' gambar</span>';
  else m.innerHTML = '<span class="warn">Paragraf: ' + paras.length + ', gambar: ' + panels.length + '. Idealnya sama.</span>';
  $('plist').innerHTML = paras.map((p, i) => '<div class="para"><b>' + (i + 1) + '.</b> ' + esc(snip(p, 140)) + '</div>').join('');
  $('plist').hidden = !paras.length;
}
function esc(s) { return s.replace(/[&<>]/g, c => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;' }[c])); }

/* ---------- audio + tap sync ---------- */
$('af').onchange = e => {
  const f = e.target.files[0]; if (!f) return;
  audioFile = f;
  if (aud.src) URL.revokeObjectURL(aud.src);
  aud.src = URL.createObjectURL(f);
  resetSync(); clearDurs(); renderSync(); renderCheck();
};
aud.addEventListener('loadedmetadata', () => { renderSync(); renderCheck(); });
aud.addEventListener('timeupdate', () => { $('clock').textContent = fmt(aud.currentTime) + ' / ' + fmt(aud.duration); });
aud.addEventListener('ended', () => { if (state === 'playing') finish(true); });

function resetSync() {
  try { aud.pause(); } catch (e) {}
  state = 'idle'; idx = 0; starts = [];
  releaseWake();
}
async function keepAwake() {
  try { if (navigator.wakeLock) wake = await navigator.wakeLock.request('screen'); } catch (e) {}
}
function releaseWake() { try { if (wake) wake.release(); } catch (e) {} wake = null; }

function renderSync() {
  const n = panels.length;
  const img = $('cur');
  if (!n) {
    img.hidden = true; $('curn').textContent = 'Belum ada gambar (tab 1)';
    $('curtxt').textContent = ''; $('nexttxt').textContent = '';
  } else {
    const i = Math.min(idx, n - 1);
    img.src = panels[i].url; img.hidden = false;
    $('curn').textContent = 'Panel ' + (i + 1) + ' dari ' + n;
    $('curtxt').textContent = paras[i] || (paras.length ? '(tidak ada paragraf untuk panel ini)' : '(naskah belum diisi)');
    $('nexttxt').textContent = (i + 1 < n && paras[i + 1]) ? 'Berikutnya: ' + snip(paras[i + 1], 90) : '';
  }
  $('clock').textContent = fmt(aud.currentTime) + ' / ' + fmt(aud.duration);
  const b = $('bTap');
  if (state === 'idle') b.textContent = 'Mulai ▶';
  else if (state === 'done') b.textContent = 'Ulangi ↺';
  else if (idx < n - 1) b.textContent = 'Panel berikutnya ▶ (' + (idx + 2) + '/' + n + ')';
  else b.textContent = 'Selesai ■';
  $('bPause').textContent = state === 'paused' ? '▶' : '⏸';
  renderDurs();
}

function tap() {
  const n = panels.length;
  if (state === 'idle') {
    if (!n) { toast('Masukkan gambar dulu (tab 1)'); return; }
    if (!audioFile) { toast('Pilih audio narasi dulu'); return; }
    if (!(aud.duration > 0)) { toast('Audio belum siap, tunggu sebentar'); return; }
    aud.currentTime = 0; starts = [0]; idx = 0; state = 'playing';
    aud.play().catch(() => toast('Audio gagal diputar'));
    keepAwake();
  } else if (state === 'playing') {
    if (idx < n - 1) {
      const lag = Math.max(0, +$('lag').value || 0);
      const t = Math.max(aud.currentTime - lag, starts[idx] + 0.1);
      starts.push(+t.toFixed(2)); idx++;
    } else finish(false);
  } else if (state === 'done') {
    resetSync(); clearDurs();
  }
  renderSync();
}
function finish(auto) {
  aud.pause();
  const n = panels.length, total = aud.duration || aud.currentTime;
  durs = panels.map((_, i) => {
    const s = starts[i] !== undefined ? starts[i] : total;
    const e = (i + 1 < n && starts[i + 1] !== undefined) ? starts[i + 1] : total;
    return Math.max(0.3, +(e - s).toFixed(2));
  });
  lowconf = [];
  state = 'done'; releaseWake();
  if (auto && starts.length < n) toast('Audio habis sebelum semua panel di-tap. Cek durasi di bawah.', 4500);
  else toast('Sync selesai. Cek durasi di bawah.');
  renderSync(); renderCheck();
}
$('bTap').addEventListener('pointerdown', e => { e.preventDefault(); tap(); });
$('bPause').addEventListener('pointerdown', e => {
  e.preventDefault();
  if (state === 'playing') { aud.pause(); state = 'paused'; }
  else if (state === 'paused') { aud.play(); state = 'playing'; keepAwake(); }
  renderSync();
});
$('bUndo').addEventListener('pointerdown', e => {
  e.preventDefault();
  if ((state === 'playing' || state === 'paused') && idx > 0) { starts.pop(); idx--; renderSync(); }
});
document.addEventListener('keydown', e => {
  if (curTab !== 'sync' || e.code !== 'Space') return;
  const t = e.target.tagName;
  if (t === 'INPUT' || t === 'TEXTAREA' || t === 'SELECT') return;
  e.preventDefault(); if (!e.repeat) tap();
});

function renderDurs() {
  const box = $('durbox');
  if (!durs.length || durs.length !== panels.length) { box.hidden = true; return; }
  box.hidden = false;
  $('durlist').innerHTML = panels.map((p, i) =>
    '<div class="row"><img src="' + p.url + '" alt=""><span>Panel ' + (i + 1) +
    (lowconf.length === panels.length && lowconf[i] ? ' <b class="warn">⚠ cek</b>' : '') +
    '<small class="mut">' + esc(snip(paras[i] || '', 50)) + '</small></span>' +
    '<button class="pv" data-p="' + i + '">▶</button>' +
    '<input type="number" step="0.1" min="0.3" data-i="' + i + '" value="' + durs[i] + '"></div>').join('');
  durTotal();
}
function durTotal() {
  const t = durs.reduce((a, b) => a + b, 0);
  $('durtot').textContent = 'Total video: ' + fmt(t) + (aud.duration ? ' · Audio: ' + fmt(aud.duration) : '');
}
$('durlist').addEventListener('input', e => {
  const i = +e.target.dataset.i, v = parseFloat(e.target.value);
  if (v > 0) { durs[i] = v; durTotal(); renderCheck(); }
});
$('bEven').onclick = () => {
  const n = panels.length;
  if (!n) { toast('Masukkan gambar dulu'); return; }
  const total = aud.duration || n * 4;
  durs = panels.map(() => +(total / n).toFixed(2));
  lowconf = [];
  renderDurs(); renderCheck();
};


$('durlist').addEventListener('click', e => {
  if (e.target.classList.contains('pv')) previewPanel(+e.target.dataset.p);
});
function previewPanel(i) {
  if (!audioFile || !(aud.duration > 0)) { toast('Pilih audio dulu'); return; }
  if (state === 'playing' || state === 'paused') { toast('Selesaikan tap-sync dulu'); return; }
  clearTimeout(pvTimer);
  let t = 0;
  for (let k = 0; k < i; k++) t += durs[k];
  aud.currentTime = t;
  aud.play().catch(() => {});
  pvTimer = setTimeout(() => aud.pause(), Math.min(durs[i] + 0.5, 20) * 1000);
}

$('bManual').onclick = () => {
  manualOpen = !manualOpen;
  $('manual').hidden = !manualOpen;
  $('bManual').textContent = manualOpen ? 'Tutup tap-sync manual' : 'Tap-sync manual (opsional)';
  $('bar').hidden = !(curTab === 'sync' && manualOpen);
  if (manualOpen) renderSync();
};

fetch('/health').then(r => r.json()).then(j => {
  $('srv').textContent = j.gpu
    ? 'Server: GPU aktif (transkripsi cepat)'
    : 'Server: CPU saja. Transkripsi lebih lambat, pakai model small/base atau ganti runtime Colab ke GPU.';
}).catch(() => {});

$('bAuto').onclick = async () => {
  if (!panels.length) { toast('Masukkan gambar dulu (tab 1)'); return; }
  if (!paras.length) { toast('Isi naskah dulu (tab 2)'); return; }
  if (paras.length !== panels.length) {
    toast('Paragraf (' + paras.length + ') harus sama dengan gambar (' + panels.length + ')', 4500); return;
  }
  if (!audioFile) { toast('Pilih audio narasi dulu'); return; }
  resetSync(); renderSync();
  const btn = $('bAuto'); btn.disabled = true;
  $('apg').hidden = false; $('apg').value = 0;
  try {
    const fd = new FormData();
    fd.append('audio', audioFile);
    fd.append('paragraphs', JSON.stringify(paras));
    fd.append('model', $('wm').value);
    fd.append('language', $('lang').value);
    $('ast').textContent = 'Mengunggah audio...';
    const r = await fetch('/autosync', { method: 'POST', body: fd });
    if (!r.ok) throw new Error('Server menolak (' + r.status + ')');
    const j = await r.json();
    const s = await new Promise((resolve, reject) => {
      const iv = setInterval(async () => {
        try {
          const s = await (await fetch('/status/' + j.job_id)).json();
          $('apg').value = s.progress || 0;
          const label = s.status === 'transcribing' ? 'Mentranskrip audio (pertama kali bisa lama karena unduh model)'
            : s.status === 'aligning' ? 'Mencocokkan ke naskah' : s.status;
          $('ast').textContent = label + ' ' + Math.round((s.progress || 0) * 100) + '%' + (s.note ? ' · ' + s.note : '');
          if (s.status === 'done') { clearInterval(iv); resolve(s); }
          else if (s.status === 'error') { clearInterval(iv); reject(new Error(s.error)); }
        } catch (e) { clearInterval(iv); reject(e); }
      }, 1500);
    });
    durs = s.durations.map(d => +d);
    lowconf = s.conf.map(c => c < 0.5);
    const low = lowconf.filter(Boolean).length;
    $('ast').innerHTML = '<span class="ok">Selesai (' + (s.device === 'cuda' ? 'GPU' : 'CPU') + ', bahasa ' + (s.language || '?') + ').</span> ' +
      (low ? '<span class="warn">' + low + ' panel perlu dicek (tanda ⚠).</span>' : 'Semua panel cocok dengan baik.') +
      ' Tap ▶ di tiap panel buat dengar hasilnya.';
    renderDurs(); renderCheck();
  } catch (e) {
    $('ast').textContent = 'Error: ' + e.message;
  }
  btn.disabled = false;
};

/* ---------- render ---------- */
function ready() {
  return panels.length > 0 && durs.length === panels.length && durs.every(d => d > 0);
}
function renderCheck() {
  const c = $('chk');
  if (!panels.length) c.innerHTML = '<span class="warn">Belum ada gambar.</span>';
  else if (!ready()) c.innerHTML = '<span class="warn">Durasi panel belum diatur. Lakukan tap-sync di tab 3.</span>';
  else c.innerHTML = '<span class="ok">Siap: ' + panels.length + ' panel, ' + fmt(durs.reduce((a, b) => a + b, 0)) + (audioFile ? ', dengan audio' : ', tanpa audio') + '</span>';
}
async function shrink(file, max) {
  max = max || 3000;
  try {
    const bm = await createImageBitmap(file);
    const s = Math.min(1, max / Math.max(bm.width, bm.height));
    if (s >= 1 && (file.type === 'image/jpeg' || file.size < 2e6)) { if (bm.close) bm.close(); return file; }
    const c = document.createElement('canvas');
    c.width = Math.round(bm.width * s); c.height = Math.round(bm.height * s);
    c.getContext('2d').drawImage(bm, 0, 0, c.width, c.height);
    if (bm.close) bm.close();
    return await new Promise(res => c.toBlob(b => res(b || file), 'image/jpeg', 0.9));
  } catch (e) { return file; }
}
$('bGo').onclick = async () => {
  if (!panels.length) { toast('Masukkan gambar dulu'); return; }
  if (!ready()) { toast('Atur durasi dulu lewat tab Sync'); return; }
  const btn = $('bGo'); btn.disabled = true;
  $('pv').hidden = true; $('dl').style.display = 'none'; $('pg').value = 0;
  try {
    const [w, h] = $('ps').value.split('x').map(Number);
    const tr = parseFloat($('ptr').value);
    const cfg = {
      width: w, height: h, fps: 24, transition_sec: tr,
      scenes: durs.map(d => ({ duration: d, motionType: $('pm').value, transitionType: tr > 0 ? 'crossfade' : 'none' }))
    };
    const fd = new FormData();
    for (let i = 0; i < panels.length; i++) {
      $('st').textContent = 'Menyiapkan gambar ' + (i + 1) + '/' + panels.length;
      const b = await shrink(panels[i].file);
      fd.append('panels', b, 'p' + String(i).padStart(3, '0') + '.jpg');
    }
    fd.append('config', JSON.stringify(cfg));
    if (audioFile) fd.append('audio', audioFile);
    $('st').textContent = 'Mengunggah...';
    const r = await fetch('/render', { method: 'POST', body: fd });
    if (!r.ok) throw new Error('Server menolak (' + r.status + ')');
    const j = await r.json();
    await new Promise((resolve, reject) => {
      const iv = setInterval(async () => {
        try {
          const s = await (await fetch('/status/' + j.job_id)).json();
          $('pg').value = s.progress || 0;
          $('st').textContent = s.status + ' ' + Math.round((s.progress || 0) * 100) + '%';
          if (s.status === 'done') {
            clearInterval(iv);
            const u = '/result/' + j.job_id;
            $('pv').src = u; $('pv').hidden = false;
            $('dl').href = u; $('dl').style.display = 'block';
            resolve();
          } else if (s.status === 'error') { clearInterval(iv); reject(new Error(s.error)); }
        } catch (e) { clearInterval(iv); reject(e); }
      }, 1500);
    });
  } catch (e) {
    $('st').textContent = 'Error: ' + e.message;
  }
  btn.disabled = false;
};

renderGrid(); renderMatch(); renderSync(); renderCheck();
</script>
</body></html>
"""


def build_app():
    from fastapi import FastAPI, UploadFile, File, Form
    from fastapi.middleware.cors import CORSMiddleware
    from fastapi.responses import FileResponse, HTMLResponse, JSONResponse

    api = FastAPI(title="Manga Recap Render Server")
    api.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

    @api.get("/", response_class=HTMLResponse)
    def home():
        return TEST_PAGE

    @api.get("/health")
    def health():
        return {"status": "ok", "gpu": has_gpu()}

    @api.post("/autosync")
    async def autosync(audio: UploadFile = File(...), paragraphs: str = Form("[]"),
                       model: str = Form("small"), language: str = Form("id")):
        try:
            paras = [str(p) for p in json.loads(paragraphs or "[]")]
        except Exception:
            return JSONResponse({"error": "paragraphs bukan JSON valid"}, status_code=400)
        if not paras:
            return JSONResponse({"error": "naskah kosong"}, status_code=400)
        model = model if model in ("base", "small", "medium") else "small"
        language = language if (language == "auto" or (language.isalpha() and len(language) <= 3)) else "id"
        job_id = uuid.uuid4().hex[:10]
        d = WORK / job_id
        d.mkdir(parents=True, exist_ok=True)
        apath = d / ("audio" + (Path(audio.filename or "a.mp3").suffix or ".mp3"))
        apath.write_bytes(await audio.read())
        JOBS[job_id] = {"status": "queued", "progress": 0.0, "error": None}
        threading.Thread(target=autosync_job, args=(job_id, apath, paras, model, language), daemon=True).start()
        return {"job_id": job_id}

    @api.post("/render")
    async def render(panels: List[UploadFile] = File(...), config: str = Form("{}"),
                     audio: Optional[UploadFile] = File(None)):
        job_id = uuid.uuid4().hex[:10]
        d = WORK / job_id
        d.mkdir(parents=True, exist_ok=True)
        paths = []
        for i, up in enumerate(panels):
            ext = Path(up.filename or "x.png").suffix or ".png"
            pth = d / f"p{i:03d}{ext}"
            pth.write_bytes(await up.read())
            paths.append(pth)
        apath = None
        if audio is not None and audio.filename:
            apath = d / ("audio" + (Path(audio.filename).suffix or ".mp3"))
            apath.write_bytes(await audio.read())
        try:
            cfg = json.loads(config or "{}")
        except Exception:
            return JSONResponse({"error": "config bukan JSON valid"}, status_code=400)
        JOBS[job_id] = {"status": "queued", "progress": 0.0, "error": None}
        threading.Thread(target=render_job, args=(job_id, paths, apath, cfg), daemon=True).start()
        return {"job_id": job_id}

    @api.get("/status/{job_id}")
    def status(job_id: str):
        return JOBS.get(job_id, {"status": "error", "progress": 0, "error": "job tidak ditemukan"})

    @api.get("/result/{job_id}")
    def result(job_id: str):
        f = WORK / job_id / "final.mp4"
        if not f.exists():
            return JSONResponse({"error": "belum selesai"}, status_code=404)
        return FileResponse(str(f), media_type="video/mp4", filename="recap.mp4")

    return api


try:
    app = build_app()
except ImportError:
    app = None

Writing /content/recap_server.py


In [5]:
# Sel 3: jalankan server + buat link publik
import subprocess, time, re, threading

subprocess.run("pkill -f uvicorn; pkill -f cloudflared", shell=True)
time.sleep(1)

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "recap_server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content", stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url, t0 = None, time.time()
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
    if time.time() - t0 > 90:
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

if url:
    print("SERVER AKTIF")
    print("Link:", url)
    print("Buka link itu di browser buat dipakai, atau tempel di app.")
    print("Jangan tutup tab Colab ini selama dipakai.")
else:
    print("Link gagal dibuat. Jalankan sel ini sekali lagi.")
    print(open("/content/server.log").read()[-1500:])

SERVER AKTIF
Link: https://walnut-chelsea-liable-talks.trycloudflare.com
Buka link itu di browser buat dipakai, atau tempel di app.
Jangan tutup tab Colab ini selama dipakai.


In [ ]:
# Sel 4: pairing Recap Render ke website Vercel
# WAJIB GANTI APP_URL dan PAIRING_SECRET sebelum menjalankan sel ini.
import os, time, uuid, secrets, threading, requests

APP_URL = "https://GANTI-DENGAN-DOMAIN-VERCEL"
PAIRING_SECRET = "GANTI_DENGAN_SECRET_YANG_SAMA_DI_VERCEL"

if not url or not str(url).startswith("https://"):
    raise RuntimeError("URL Cloudflare belum tersedia. Jalankan Sel 3 terlebih dahulu.")

NOTEBOOK_ID = str(uuid.uuid4())
ACCESS_CODE = f"{secrets.randbelow(1_000_000):06d}"
PAIRING_HEADERS = {"x-pairing-secret": PAIRING_SECRET}

def register_recap_worker():
    response = requests.post(
        f"{APP_URL.rstrip('/')}/api/register",
        headers=PAIRING_HEADERS,
        json={
            "code": ACCESS_CODE,
            "notebook_id": NOTEBOOK_ID,
            "url": url,
            "service": "recap_render",
        },
        timeout=20,
    )
    response.raise_for_status()
    return response.json()

register_recap_worker()

def recap_worker_heartbeat():
    while True:
        try:
            response = requests.post(
                f"{APP_URL.rstrip('/')}/api/heartbeat",
                headers=PAIRING_HEADERS,
                json={"notebook_id": NOTEBOOK_ID, "url": url},
                timeout=20,
            )
            if response.status_code == 404:
                register_recap_worker()
            else:
                response.raise_for_status()
        except Exception as exc:
            print("Heartbeat pairing belum berhasil:", exc)
        time.sleep(30)

threading.Thread(target=recap_worker_heartbeat, daemon=True).start()

print("=" * 48)
print("RECAP RENDER WORKER TERDAFTAR")
print("KODE AKSES:", ACCESS_CODE)
print("Worker ID:", NOTEBOOK_ID)
print("Status: heartbeat berjalan setiap 30 detik")
print("Masukkan kode ini di website Recap Render.")
print("Jangan tutup runtime Colab saat digunakan.")
print("=" * 48)
